# EfficientNet-B0 Colab Training

Research-only. Non-commercial use only. Not for diagnosis, treatment, or clinical deployment. Doctor review required.

This notebook trains Milestone 1: broad dermatology main-class prediction on DermaCon-IN using ImageNet-pretrained EfficientNet-B0.

## 1. Runtime

In Colab: `Runtime` -> `Change runtime type` -> choose a GPU such as T4.

In [ ]:
!nvidia-smi

## 2. Mount Drive And Set Dataset Source

Dataset source:

```text
https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/W7OUZM
DOI: 10.7910/DVN/W7OUZM
```

Drive is still used to persist downloaded data and training outputs across Colab sessions.

Recommended Drive layout after download:

```text
MyDrive/derm-opd-triage/data/raw/DATASET/*.jpg
MyDrive/derm-opd-triage/data/raw/METADATA/Skin_Metadata.csv
MyDrive/derm-opd-triage/data/raw/train_split.csv  # optional
MyDrive/derm-opd-triage/data/raw/test_split.csv   # optional
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/derm-opd-triage'
REPO_URL = 'https://github.com/Abhigyan-Shekhar/skin-lesion-detect-model.git'
REPO_DIR = '/content/skin-lesion-detect-model'
DATAVERSE_PID = 'doi:10.7910/DVN/W7OUZM'

## 3. Clone Repo And Install Dependencies

In [ ]:
import os
from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 4. Download Or Link DermaCon-IN Data

The notebook first checks `MyDrive/derm-opd-triage/data/raw`. If that cache is empty, it downloads files from Harvard Dataverse using DOI `10.7910/DVN/W7OUZM`, then symlinks the Drive data into the repo.

If Dataverse throttles large downloads, manually place the files in the Drive layout and rerun this cell.

In [ ]:
from pathlib import Path
import shutil

drive_raw = Path(DRIVE_PROJECT) / 'data' / 'raw'
repo_data = Path(REPO_DIR) / 'data'
repo_raw = repo_data / 'raw'
repo_splits = repo_data / 'splits'
repo_data.mkdir(parents=True, exist_ok=True)
repo_splits.mkdir(parents=True, exist_ok=True)

if drive_raw.exists() and any(drive_raw.iterdir()):
    print('Using existing Drive dataset cache:', drive_raw)
else:
    print('Downloading DermaCon-IN from Harvard Dataverse into Drive cache...')
    drive_raw.mkdir(parents=True, exist_ok=True)
    !python src/download_dataverse.py --persistent_id {DATAVERSE_PID} --output_dir {drive_raw}

if repo_raw.exists() or repo_raw.is_symlink():
    if repo_raw.is_symlink():
        repo_raw.unlink()
    else:
        shutil.rmtree(repo_raw)

repo_raw.symlink_to(drive_raw, target_is_directory=True)
print('Linked', repo_raw, '->', drive_raw)
!find data/raw -maxdepth 3 -type f | head -40

## 5. Inspect Metadata

In [ ]:
!python src/inspect_metadata.py \
  --metadata data/raw/METADATA/Skin_Metadata.csv \
  --image_dir data/raw/DATASET

## 6. Prepare Splits

If patient IDs exist, splitting is patient-level. If official `train_split.csv` and `test_split.csv` exist under `data/raw`, they are used.

In [ ]:
!python src/prepare_splits.py \
  --metadata data/raw/METADATA/Skin_Metadata.csv \
  --output_dir data/splits

!cat data/splits/split_summary.json

## 7. Train EfficientNet-B0

Default: 10 frozen-head epochs + 20 fine-tuning epochs. If you need a quick smoke run, edit `configs/efficientnet_b0.yaml` and reduce `epochs_head` / `epochs_finetune`.

In [ ]:
!python src/train.py --config configs/efficientnet_b0.yaml

## 8. Evaluate Best Checkpoint

In [ ]:
!python src/evaluate.py \
  --checkpoint outputs/checkpoints/best.pt \
  --split data/splits/test.csv

!cat outputs/metrics/metrics.json
!ls -R outputs | sed -n '1,120p'

## 9. Save Artifacts Back To Drive

In [ ]:
from pathlib import Path
import shutil

drive_outputs = Path(DRIVE_PROJECT) / 'outputs'
drive_outputs.mkdir(parents=True, exist_ok=True)

for folder in ['checkpoints', 'metrics', 'plots', 'predictions']:
    src = Path(REPO_DIR) / 'outputs' / folder
    dst = drive_outputs / folder
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('Copied', src, '->', dst)

!find {DRIVE_PROJECT}/outputs -maxdepth 3 -type f | sort

## 10. Optional Inference Test

In [ ]:
# Replace with an actual image path from your dataset.
# !python src/inference.py --checkpoint outputs/checkpoints/best.pt --image data/raw/DATASET/example.jpg